# 序列逆置
使用sequence to sequence 模型将一个字符串序列逆置。
例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个sequence to sequence 模型示意图 )
![seq2seq](./seq2seq.png)

In [20]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [21]:
import random
import string

def randomString(stringLength):
    """
    Generate a random string with the combination of lowercase and uppercase letters
    随机生成一个由大小写字母组成的字符串
    """

    letters = string.ascii_uppercase 
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    '''生成一个batch的训练数据
    batch_size: 每个batch的样本数量
    length: 每个样本的字符串长度
    '''
    batched_examples = [randomString(length) for i in range(batch_size)]

    # 将字符串转换为数字，A->1, B->2, ..., Z->26
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    # 将输入字符串反转，作为输出字符串
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    # 在输出字符串的开头添加一个特殊的开始标记0，作为解码器的输入
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['DPLSEKZBWX', 'NIUGHCQCNO'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 4, 16, 12, 19,  5, 11, 26,  2, 23, 24],
       [14,  9, 21,  7,  8,  3, 17,  3, 14, 15]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 24, 23,  2, 26, 11,  5, 19, 12, 16],
       [ 0, 15, 14,  3, 17,  3,  8,  7, 21,  9]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[24, 23,  2, 26, 11,  5, 19, 12, 16,  4],
       [15, 14,  3, 17,  3,  8,  7, 21,  9, 14]], dtype=int32)>)


# 建立sequence to sequence 模型

In [ ]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27 # vocab size, 包含26个字母和一个特殊的开始标记0
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64) # embedding层，将输入的整数序列转换为稠密的向量表示，输出形状为(batch_size, seq_len, emb_sz)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(128)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(128)
        
        # 编码器的RNN层，使用SimpleRNNCell作为单元，返回每个时间步的输出和最后的状态
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        # 解码器的RNN层，使用SimpleRNNCell作为单元，返回每个时间步的输出和最后的状态
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(self.v_sz) # 输出层，输出每个时间步的预测结果，形状为(batch_size, seq_len, vocab_size)
        
    # @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        完成 sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好
        env_ids: 编码器的输入，形状为(batch_size, seq_len)
        dec_ids: 解码器的输入，形状为(batch_size, seq_len)
        '''
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz),将输入的整数序列转换为稠密的向量表示
        # 编码器的RNN层，使用SimpleRNNCell作为单元，返回每个时间步的输出和最后的状态
        enc_out, enc_state = self.encoder(enc_emb) # enc_out shape(b_sz, len, h_sz), enc_state shape(b_sz, h_sz)
        
        dec_emb = self.embed_layer(dec_ids) # shape(b_sz, len, emb_sz) ，将解码器的输入整数序列转换为稠密的向量表示
        # 解码器的RNN层，使用SimpleRNNCell作为单元，返回每个时间步的输出和最后的状态，初始状态为编码器的最后状态
        dec_out, _ = self.decoder(dec_emb, initial_state=enc_state) # dec_out shape(b_sz, len, h_sz)

        # 输出层，输出每个时间步的预测结果，形状为(batch_size, seq_len, vocab_size)
        logits = self.dense(dec_out) # shape(b_sz, len, v_sz)

        return logits
    
    
#     @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        
        return [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,] 
        x是当前时间步的输入，state是当前时间步的状态
        获取下一个时间步的预测结果，输入为当前时间步的输入和状态，输出为下一个时间步的预测结果和状态
        '''
        # 将输入的整数序列转换为稠密的向量表示
        inp_emb = self.embed_layer(x) #shape(b_sz, emb_sz)
        # 生成下一个时间步的输出和状态
        h, state = self.decoder_cell.call(inp_emb, state) # shape(b_sz, h_sz)
        # 输出层，输出每个时间步的预测结果，形状为(batch_size, vocab_size)
        logits = self.dense(h) # shape(b_sz, v_sz)
        #将输出层的预测结果转化为整数序列，取最大值的索引作为预测结果
        out = tf.argmax(logits, axis=-1)
        return out, state

# Loss函数以及训练逻辑

In [23]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(3000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [24]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.303262
step 500 : loss 1.6601282
step 1000 : loss 1.0435333
step 1500 : loss 0.7263564
step 2000 : loss 0.58017147
step 2500 : loss 0.4168233


<tf.Tensor: shape=(), dtype=float32, numpy=0.35556912422180176>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [25]:
def sequence_reversal():
    def decode(init_state, steps=10):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 10)
    state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1]), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
[('SODWTPDZBM', 'MBZDPTWDOS'), ('RRJOEDYUUZ', 'ZUUYDEOJRR'), ('YWLMZVMFDX', 'XDFMVZMLWY'), ('CZIIKDWKRE', 'ERKWDKIIZC'), ('KVQACDXOWZ', 'ZWOXDCAQVK'), ('GTVRBUXJFI', 'IFJXUBRVTG'), ('ZVZFSAMLYT', 'TYLMASFZVZ'), ('ZKBVPXSVTV', 'VTVSXPVBKZ'), ('WTGESZXAFK', 'KFAXZSEGTW'), ('QTEZSZJJCU', 'UCJJZSZETQ'), ('DXDMCGQZAE', 'EAZQGCMDXD'), ('KLQHEZBXZV', 'VZXBZEHQLK'), ('UHEMHUOEVX', 'XVEOUHMEHU'), ('KCAMKERDHL', 'LHDREKMACK'), ('LBDLZEACKQ', 'QKCAEZLDBL'), ('JXCOFAPJTM', 'MTJPAFOCXJ'), ('AXPJQDMQBA', 'ABQMDQJPXA'), ('DFKNNZZMKU', 'UKMZZNNKFD'), ('SGGQTEEASE', 'ESAEETQGGS'), ('PZQUJYDOCO', 'OCODYJUQZP'), ('KTXVFODJJQ', 'QJJDOFVXTK'), ('CUVEQRFEXP', 'PXEFRQEVUC'), ('EUIXEHQBVJ', 'JVBQHEXIUE'), ('VCFOLVADZZ', 'ZZDAVLOFCV'), ('OPGKZIJXKR', 'RKXJIZKGPO'), ('WKXTTTFDFP', 'PFDFTTTXKW'), ('DWLTMSBJCB', 'BCJBSMTLWD